In [1]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../data/raw")

df_scores = pd.read_csv(
    DATA_PATH / "2010-2025_scores.csv"
)

print(df_scores.shape)

df_scores.head()

(5832, 9)


,Season,Week,GameStatus,GameSlot,GameDate,AwayTeam,AwayScore,HomeTeam,HomeScore
0,2010,HALL OF FAME,FINAL,Sunday,August 8th,DAL,16.0,CIN,7.0
1,2010,PRESEASON WEEK 1,FINAL,Thursday,August 12th,CAR,12.0,BAL,17.0
2,2010,PRESEASON WEEK 1,FINAL,Thursday,August 12th,NO,24.0,NE,27.0
3,2010,PRESEASON WEEK 1,FINAL,Thursday,August 12th,OAK,17.0,DAL,9.0
4,2010,PRESEASON WEEK 1,FINAL,Friday,August 13th,BUF,17.0,WAS,42.0


In [2]:
df_scores.columns.tolist()

['Season',
 'Week',
 'GameStatus',
 'GameSlot',
 'GameDate',
 'AwayTeam',
 'AwayScore',
 'HomeTeam',
 'HomeScore']

In [3]:
df_scores["Week"].unique()

<StringArray>
[            'HALL OF FAME',         'PRESEASON WEEK 1',
         'PRESEASON WEEK 2',         'PRESEASON WEEK 3',
         'PRESEASON WEEK 4',                   'WEEK 1',
                   'WEEK 2',                   'WEEK 3',
                   'WEEK 4',                   'WEEK 5',
                   'WEEK 6',                   'WEEK 7',
                   'WEEK 8',                   'WEEK 9',
                  'WEEK 10',                  'WEEK 11',
                  'WEEK 12',                  'WEEK 13',
                  'WEEK 14',                  'WEEK 15',
                  'WEEK 16',                  'WEEK 17',
        'WILD CARD WEEKEND',      'DIVISIONAL PLAYOFFS',
 'CONFERENCE CHAMPIONSHIPS',                 'PRO BOWL',
               'SUPER BOWL',                  'WEEK 18']
Length: 28, dtype: str

In [4]:
exclude_weeks = [
    "HALL OF FAME",
    "PRESEASON WEEK 1",
    "PRESEASON WEEK 2",
    "PRESEASON WEEK 3",
    "PRESEASON WEEK 4",
    "PRO BOWL"
]

df_clean = df_scores[
    ~df_scores["Week"].isin(exclude_weeks)
].copy()

print(df_clean.shape)

df_clean["Week"].unique()

(4926, 9)


<StringArray>
[                  'WEEK 1',                   'WEEK 2',
                   'WEEK 3',                   'WEEK 4',
                   'WEEK 5',                   'WEEK 6',
                   'WEEK 7',                   'WEEK 8',
                   'WEEK 9',                  'WEEK 10',
                  'WEEK 11',                  'WEEK 12',
                  'WEEK 13',                  'WEEK 14',
                  'WEEK 15',                  'WEEK 16',
                  'WEEK 17',        'WILD CARD WEEKEND',
      'DIVISIONAL PLAYOFFS', 'CONFERENCE CHAMPIONSHIPS',
               'SUPER BOWL',                  'WEEK 18']
Length: 22, dtype: str

In [5]:
df_clean["HomeWin"] = (
    df_clean["HomeScore"] > df_clean["AwayScore"]
).astype(int)

df_clean[[
    "Season",
    "Week",
    "HomeTeam",
    "AwayTeam",
    "HomeScore",
    "AwayScore",
    "HomeWin"
]].head()

,Season,Week,HomeTeam,AwayTeam,HomeScore,AwayScore,HomeWin
65,2010,WEEK 1,NO,MIN,14.0,9.0,1
66,2010,WEEK 1,TB,CLE,17.0,14.0,1
67,2010,WEEK 1,BUF,MIA,10.0,15.0,0
68,2010,WEEK 1,NE,CIN,38.0,24.0,1
69,2010,WEEK 1,HOU,IND,34.0,24.0,1


In [6]:
home_df = df_clean[[
    "Season",
    "Week",
    "HomeTeam",
    "AwayTeam",
    "HomeScore",
    "AwayScore",
    "HomeWin"
]].copy()

home_df.columns = [
    "Season",
    "Week",
    "Team",
    "Opponent",
    "PointsFor",
    "PointsAgainst",
    "Win"
]

away_df = df_clean[[
    "Season",
    "Week",
    "AwayTeam",
    "HomeTeam",
    "AwayScore",
    "HomeScore",
    "HomeWin"
]].copy()

away_df.columns = [
    "Season",
    "Week",
    "Team",
    "Opponent",
    "PointsFor",
    "PointsAgainst",
    "HomeWinOpponent"
]

# Si ganó local, perdió visitante
away_df["Win"] = 1 - away_df["HomeWinOpponent"]

away_df = away_df.drop(columns=["HomeWinOpponent"])

team_games = pd.concat(
    [home_df, away_df],
    ignore_index=True
)

team_games.head(10)

,Season,Week,Team,Opponent,PointsFor,PointsAgainst,Win
0,2010,WEEK 1,NO,MIN,14.0,9.0,1
1,2010,WEEK 1,TB,CLE,17.0,14.0,1
2,2010,WEEK 1,BUF,MIA,10.0,15.0,0
3,2010,WEEK 1,NE,CIN,38.0,24.0,1
4,2010,WEEK 1,HOU,IND,34.0,24.0,1
5,2010,WEEK 1,JAC,DEN,24.0,17.0,1
6,2010,WEEK 1,TEN,OAK,38.0,13.0,1
7,2010,WEEK 1,NYG,CAR,31.0,18.0,1
8,2010,WEEK 1,CHI,DET,19.0,14.0,1
9,2010,WEEK 1,STL,ARI,13.0,17.0,0


In [7]:
team_games = team_games.sort_values(
    by=["Season", "Week"]
).reset_index(drop=True)

team_games.head()

,Season,Week,Team,Opponent,PointsFor,PointsAgainst,Win
0,2010,CONFERENCE CHAMPIONSHIPS,PIT,NYJ,24.0,19.0,1
1,2010,CONFERENCE CHAMPIONSHIPS,CHI,GB,14.0,21.0,0
2,2010,CONFERENCE CHAMPIONSHIPS,NYJ,PIT,19.0,24.0,0
3,2010,CONFERENCE CHAMPIONSHIPS,GB,CHI,21.0,14.0,1
4,2010,DIVISIONAL PLAYOFFS,PIT,BAL,31.0,24.0,1


In [8]:
team_games["AvgPointsLast5"] = (
    team_games
    .groupby("Team")["PointsFor"]
    .transform(
        lambda x: x.shift(1).rolling(5).mean()
    )
)

team_games[[
    "Season",
    "Week",
    "Team",
    "PointsFor",
    "AvgPointsLast5"
]].head(15)

,Season,Week,Team,PointsFor,AvgPointsLast5
0,2010,CONFERENCE CHAMPIONSHIPS,PIT,24.0,NaN
1,2010,CONFERENCE CHAMPIONSHIPS,CHI,14.0,NaN
2,2010,CONFERENCE CHAMPIONSHIPS,NYJ,19.0,NaN
3,2010,CONFERENCE CHAMPIONSHIPS,GB,21.0,NaN
4,2010,DIVISIONAL PLAYOFFS,PIT,31.0,NaN
5,2010,DIVISIONAL PLAYOFFS,ATL,21.0,NaN
6,2010,DIVISIONAL PLAYOFFS,NE,21.0,NaN
7,2010,DIVISIONAL PLAYOFFS,CHI,35.0,NaN
8,2010,DIVISIONAL PLAYOFFS,BAL,24.0,NaN
9,2010,DIVISIONAL PLAYOFFS,GB,48.0,NaN


In [9]:
week_order = {
    "WEEK 1": 1,
    "WEEK 2": 2,
    "WEEK 3": 3,
    "WEEK 4": 4,
    "WEEK 5": 5,
    "WEEK 6": 6,
    "WEEK 7": 7,
    "WEEK 8": 8,
    "WEEK 9": 9,
    "WEEK 10": 10,
    "WEEK 11": 11,
    "WEEK 12": 12,
    "WEEK 13": 13,
    "WEEK 14": 14,
    "WEEK 15": 15,
    "WEEK 16": 16,
    "WEEK 17": 17,
    "WEEK 18": 18,
    "WILD CARD WEEKEND": 19,
    "DIVISIONAL PLAYOFFS": 20,
    "CONFERENCE CHAMPIONSHIPS": 21,
    "SUPER BOWL": 22,
}

team_games["WeekNumber"] = team_games["Week"].map(week_order)

team_games = team_games.sort_values(
    by=["Season", "WeekNumber"]
).reset_index(drop=True)

team_games[["Season", "Week", "WeekNumber", "Team", "PointsFor", "PointsAgainst", "Win"]].head(20)

,Season,Week,WeekNumber,Team,PointsFor,PointsAgainst,Win
0,2010,WEEK 1,1,NO,14.0,9.0,1
1,2010,WEEK 1,1,TB,17.0,14.0,1
2,2010,WEEK 1,1,BUF,10.0,15.0,0
3,2010,WEEK 1,1,NE,38.0,24.0,1
4,2010,WEEK 1,1,HOU,34.0,24.0,1
5,2010,WEEK 1,1,JAC,24.0,17.0,1
6,2010,WEEK 1,1,TEN,38.0,13.0,1
7,2010,WEEK 1,1,NYG,31.0,18.0,1
8,2010,WEEK 1,1,CHI,19.0,14.0,1
9,2010,WEEK 1,1,STL,13.0,17.0,0


In [10]:
team_games["AvgPointsLast5"] = (
    team_games
    .groupby("Team")["PointsFor"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

team_games[[
    "Season",
    "Week",
    "Team",
    "PointsFor",
    "AvgPointsLast5"
]].head(40)

,Season,Week,Team,PointsFor,AvgPointsLast5
0,2010,WEEK 1,NO,14.0,NaN
1,2010,WEEK 1,TB,17.0,NaN
2,2010,WEEK 1,BUF,10.0,NaN
3,2010,WEEK 1,NE,38.0,NaN
4,2010,WEEK 1,HOU,34.0,NaN
5,2010,WEEK 1,JAC,24.0,NaN
6,2010,WEEK 1,TEN,38.0,NaN
7,2010,WEEK 1,NYG,31.0,NaN
8,2010,WEEK 1,CHI,19.0,NaN
9,2010,WEEK 1,STL,13.0,NaN


In [11]:
team_games[
    team_games["Team"] == "KC"
][[
    "Season",
    "Week",
    "PointsFor",
    "AvgPointsLast5"
]].head(10)

,Season,Week,PointsFor,AvgPointsLast5
15,2010,WEEK 1,21.0,NaN
51,2010,WEEK 2,16.0,NaN
73,2010,WEEK 3,31.0,NaN
129,2010,WEEK 4,NaN,NaN
152,2010,WEEK 5,9.0,NaN
190,2010,WEEK 6,31.0,NaN
207,2010,WEEK 7,42.0,NaN
243,2010,WEEK 8,13.0,NaN
307,2010,WEEK 9,20.0,NaN
342,2010,WEEK 10,29.0,23.0


In [12]:
team_games["AvgAllowedLast5"] = (
    team_games
    .groupby("Team")["PointsAgainst"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

team_games[
    team_games["Team"] == "KC"
][[
    "Season",
    "Week",
    "PointsFor",
    "PointsAgainst",
    "AvgPointsLast5",
    "AvgAllowedLast5"
]].head(10)

,Season,Week,PointsFor,PointsAgainst,AvgPointsLast5,AvgAllowedLast5
15,2010,WEEK 1,21.0,14.0,NaN,NaN
51,2010,WEEK 2,16.0,14.0,NaN,NaN
73,2010,WEEK 3,31.0,10.0,NaN,NaN
129,2010,WEEK 4,NaN,NaN,NaN,NaN
152,2010,WEEK 5,9.0,19.0,NaN,NaN
190,2010,WEEK 6,31.0,35.0,NaN,NaN
207,2010,WEEK 7,42.0,20.0,NaN,NaN
243,2010,WEEK 8,13.0,10.0,NaN,NaN
307,2010,WEEK 9,20.0,23.0,NaN,NaN
342,2010,WEEK 10,29.0,49.0,23.0,21.4


In [13]:
team_games[
    (team_games["Team"] == "KC") &
    (team_games["Week"] == "WEEK 4")
]

,Season,Week,Team,Opponent,PointsFor,PointsAgainst,Win,AvgPointsLast5,WeekNumber,AvgAllowedLast5
129,2010,WEEK 4,KC,NaN,NaN,NaN,1,NaN,4,NaN
710,2011,WEEK 4,KC,MIN,22.0,17.0,1,8.8,4,34.0
1311,2012,WEEK 4,KC,SD,20.0,37.0,0,17.6,4,23.6
1920,2013,WEEK 4,KC,NYG,31.0,7.0,1,17.4,4,18.4
2532,2014,WEEK 4,KC,NE,41.0,14.0,1,25.8,4,27.4
3147,2015,WEEK 4,KC,CIN,21.0,36.0,0,22.0,4,23.2
3762,2016,WEEK 4,KC,PIT,14.0,43.0,0,23.8,4,15.2
4351,2017,WEEK 4,KC,WAS,29.0,20.0,1,NaN,4,NaN
4971,2018,WEEK 4,KC,DEN,27.0,23.0,1,33.2,4,27.6
5567,2019,WEEK 4,KC,DET,34.0,30.0,1,32.6,4,22.8
